In [1]:
import numpy as np
import torch

In [2]:
X = np.array([160, 170, 180, 190])
y = np.array([0, 0, 1, 1])
print(X)
print(y)

[160 170 180 190]
[0 0 1 1]


In [3]:
X_mean = np.mean(X)
X_std = np.std(X)
X_norm = (X - X_mean) / X_std
print(X_mean)
print(X_std)
print(X_norm)

175.0
11.180339887498949
[-1.34164079 -0.4472136   0.4472136   1.34164079]


In [4]:
X_norm_tensor = torch.tensor(X_norm, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

# torch.nn.Linear(1, 1)에 넣으려면 각 데이터가 '입력 특성 1개'를 가진 형태,
# 즉 shape(n, 1) 이어야 합니다. 그래서 reshape(-1, 1)로 모양을 바꿉니다.
# -1 : 행 개수는 알아서 (여기서는 4)
#  1 : 열 개수는 1개 (입력 특성 1개)
X_norm_tensor = X_norm_tensor.reshape(-1, 1)
y_tensor = y_tensor.reshape(-1, 1)

print("학습용 입력 tensor X_norm_tensor:\n", X_norm_tensor)
print("학습용 정답 tensor y_tensor:\n", y_tensor)

# shape를 꼭 확인. 둘 다 (4, 1)이어야 합니다
print("X_norm_tensor shape:", X_norm_tensor.shape)
print("y_tensor shape:", y_tensor.shape)

학습용 입력 tensor X_norm_tensor:
 tensor([[-1.3416],
        [-0.4472],
        [ 0.4472],
        [ 1.3416]])
학습용 정답 tensor y_tensor:
 tensor([[0.],
        [0.],
        [1.],
        [1.]])
X_norm_tensor shape: torch.Size([4, 1])
y_tensor shape: torch.Size([4, 1])


In [5]:
# torch.nn.Linear(1, 1) 생성 (기존 a, b를 직접 만들지 않습니다.)

# torch.manual_seed(42)
#  - PyTorch의 랜덤 초기값을 고정합니다.
#  - nn.Linear는 weight(=a)와 bias(=b)를 랜덤으로 초기화하므로,
#    seed를 고정해야 실행할 때마다 결과가 크게 달라지지 않습니다.
#  - 반드시 linear를 만들기 '직전'에 둡니다.
torch.manual_seed(42)

# torch.nn.Linear(1, 1)
#  - 입력 특성 1개(키 하나)를 받아 출력 1개(H(x) 값 하나)를 내보내는 부품입니다.
#  - 내부적으로 H(x) = a·X_norm + b 를 계산합니다.
#  - 기존 강의의 a는 linear.weight, 기존 강의의 b는 linear.bias 입니다.
#  - 이전 파일처럼 a, b tensor를 직접 만들지 않는다는 점이 핵심입니다.
linear = torch.nn.Linear(1, 1)

# linear 안에 자동으로 만들어진 weight(=a)와 bias(=b)를 확인합니다.
# 학생들이 'a, b가 사라졌다'고 느끼지 않도록 학습 '전'의 초기값을 직접 봐 둡니다.
print("linear.weight:", linear.weight)  # 기존 강의의 a 역할
print("linear.bias:", linear.bias)      # 기존 강의의 b 역할

linear.weight: Parameter containing:
tensor([[0.7645]], requires_grad=True)
linear.bias: Parameter containing:
tensor([0.8300], requires_grad=True)


In [6]:
# torch.nn.BCELoss()
#  - Binary Cross Entropy Cost를 계산해 주는 PyTorch 부품입니다.
#  - 사용법: mean_cost = criterion(z, y_tensor)
#  - 주의: 입력으로 H가 아니라, sigmoid를 통과한 예측 확률 z를 넣어야 합니다!
#    (H를 넣는 BCEWithLogitsLoss는 이번 노트북에서 사용하지 않습니다.)
#  - 변수 이름은 관례적으로 criterion(판정 기준)이라고 짓습니다.
criterion = torch.nn.BCELoss()

print("criterion 준비 완료:", criterion)

criterion 준비 완료: BCELoss()


In [7]:
# learning_rate(학습률): 한 번에 weight, bias를 얼마나 크게 수정할지 정하는 값입니다.
# 이 값은 바로 다음 셀에서 optimizer를 만들 때 lr=learning_rate 로 넘겨줍니다.
learning_rate = 0.1

# epochs(에폭): 전체 데이터를 몇 번 반복해서 학습할지 정하는 값입니다.
# 여기서는 같은 데이터로 1000번 반복 학습합니다.
epochs = 1000

print("learning_rate:", learning_rate)
print("epochs:", epochs)

learning_rate: 0.1
epochs: 1000


In [8]:
# torch.optim.SGD(linear.parameters(), lr=learning_rate)
# - linear.parameters(): optimizer가 업데이트할 학습 대상(weight와 bias)을 가져옵니다.
# - lr=learning_rate: 한 번에 얼마나 움직일지(학습률)를 지정합니다.
optimizer = torch.optim.SGD(linear.parameters(), lr=learning_rate)

# 학습 대상 확인하기
# linear.parameters()가 실제로 어떤 파라미터들을 가져오는지 리스트로 출력합니다.
print(list(linear.parameters()))

[Parameter containing:
tensor([[0.7645]], requires_grad=True), Parameter containing:
tensor([0.8300], requires_grad=True)]


In [10]:
for epoch in range(epochs):
    # 1. 이전 epoch의 기울기(gradient) 초기화
    # PyTorch는 grad를 누적(더하기)하므로 매번 0으로 초기화해야 합니다.
    optimizer.zero_grad()
    
    # 2. 모델 예측값(H) 계산 (선형 연산: a*X + b)
    H = linear(X_norm_tensor)
    
    # 3. 예측 확률(z) 계산 (Sigmoid 적용)
    z = torch.sigmoid(H)
    
    # 4. 비용(Cost) 계산 (Binary Cross Entropy)
    mean_cost = criterion(z, y_tensor)
    
    # 5. 역전파(Backpropagation) 수행
    # 기울기(grad)를 자동으로 계산합니다.
    mean_cost.backward()
    
    # 6. 가중치 업데이트
    # 계산된 기울기를 바탕으로 weight와 bias를 수정합니다.
    optimizer.step()

    # 7. 학습 상태 출력
    # 100 epoch마다 한 번씩, 그리고 마지막 epoch에서 결과를 출력합니다.
    if epoch % 100 == 0 or epoch == epochs - 1:
        print(
            f"epoch={epoch}, "
            f"Cost={mean_cost.item():.6f}, "
            f"weight(a)={linear.weight.item():.6f}, "
            f"bias(b)={linear.bias.item():.6f}"
        )

    # (참고) 초반 3 epoch에서만 기울기(grad) 값을 확인합니다.
    # 기존 autograd 방식과 달리 linear 모델의 가중치/편향 grad를 확인합니다.
    if epoch < 3:
        print(
            f" (확인용) linear.weight.grad={linear.weight.grad.item():.6f}, "
            f"linear.bias.grad={linear.bias.grad.item():.6f}"
        )

epoch=0, Cost=0.483993, weight(a)=0.822395, bias(b)=0.795537
 (확인용) linear.weight.grad=-0.286153, linear.bias.grad=0.169918
 (확인용) linear.weight.grad=-0.280072, linear.bias.grad=0.165171
 (확인용) linear.weight.grad=-0.274170, linear.bias.grad=0.160554
epoch=100, Cost=0.177820, weight(a)=2.299029, bias(b)=0.171162
epoch=200, Cost=0.125015, weight(a)=3.008020, bias(b)=0.061027
epoch=300, Cost=0.099084, weight(a)=3.513456, bias(b)=0.026634
epoch=400, Cost=0.082766, weight(a)=3.915925, bias(b)=0.013142
epoch=500, Cost=0.071300, weight(a)=4.253736, bias(b)=0.007075
epoch=600, Cost=0.062714, weight(a)=4.546238, bias(b)=0.004070
epoch=700, Cost=0.056008, weight(a)=4.804815, bias(b)=0.002468
epoch=800, Cost=0.050611, weight(a)=5.036851, bias(b)=0.001564
epoch=900, Cost=0.046166, weight(a)=5.247462, bias(b)=0.001027
epoch=999, Cost=0.042473, weight(a)=5.438509, bias(b)=0.000699


In [11]:
# 1000번 반복 학습된 최종 가중치와 편향을 확인합니다.
# .item()을 사용하여 텐서 안의 순수 숫자 값만 추출합니다.
print("학습된 weight(a):", linear.weight.item())
print("학습된 bias(b):", linear.bias.item())

# 텐서의 원래 형태를 확인합니다.
# shape(크기)와 requires_grad(기울기 계산 필요 여부) 정보가 포함되어 있습니다.
print("linear.weight:", linear.weight)
print("linear.bias:", linear.bias)

학습된 weight(a): 5.438509464263916
학습된 bias(b): 0.0006986978114582598
linear.weight: Parameter containing:
tensor([[5.4385]], requires_grad=True)
linear.bias: Parameter containing:
tensor([0.0007], requires_grad=True)


In [13]:
# 새로운 입력값 예측
# 키가 185cm인 사람이 농구선수인지 예측합니다.
input_height = 185

# 새로운 입력값도 학습 데이터와 '같은 기준'으로 정규화해야 합니다.
# 학습 때 사용한 X_mean, X_std 를 그대로 다시 사용합니다. (새로 계산하면 안 됩니다.)
input_norm = (input_height - X_mean) / X_std

# 예측은 학습이 아니므로 미분값을 계산할 필요가 없습니다.
# 그래서 torch.no_grad() 안에서 계산해, 불필요한 미분 기록을 만들지 않습니다.
with torch.no_grad():
    # torch.nn.Linear(1, 1)에 넣으려면 입력 shape을 (1, 1)로 맞춰야 합니다.
    # [[input_norm]]: 이중 대괄호로 감싸 (데이터 1개, 입력 특성 1개) = (1, 1) 형태로 만듭니다.
    input_norm_tensor = torch.tensor([[input_norm]], dtype=torch.float32)
    print("input_norm_tensor shape:", input_norm_tensor.shape)
    
    # H(x) = a * X_norm + b (학습된 linear가 계산 - 선형 계산값이며 확률이 아닙니다.)
    H_new = linear(input_norm_tensor)
    # z = sigmoid(H) (예측 확률 - 0~1사이)
    z_new = torch.sigmoid(H_new)
    # 이진 분류에서는 보통 확률이 0.5 이상이면 1, 0.5 미만이면 0으로 판단합니다.
    pred = 1 if z_new.item() >= 0.5 else 0

# 출력/판정할 때는 .item()으로 tensor에서 숫자만 꺼냅니다.
print(f"\n키가 {input_height}cm인 사람의 농구선수 확률: { z_new.item():.4f}")

if pred == 1:
    print("판별 결과: 농구선수입니다.")
else:
    print("판별 결과: 농구선수가 아닙니다.")

input_norm_tensor shape: torch.Size([1, 1])

키가 185cm인 사람의 농구선수 확률: 0.9923
판별 결과: 농구선수입니다.
